# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [82]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier






In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


### Cleaning the DataFrame 

In [43]:
#your code here

# Drops any row containing at least one NaN value

spaceship_clean = spaceship.dropna().copy()

# Check how many rows you lost:

print("Original rows:", len(spaceship))
print("Remaining rows:", len(spaceship_clean))

# Extract the first letter before the slash to get the Deck

spaceship_clean['Cabin'] = spaceship_clean['Cabin'].str.split('/').str[0]
spaceship_clean.head()
#dropping the columns :
spaceship_clean = spaceship_clean.drop(columns=['PassengerId', 'Name'])
spaceship_clean.head()

Original rows: 8693
Remaining rows: 6606


,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [37]:
from sklearn.preprocessing import StandardScaler

# 1. Define the specific continuous numerical features to scale
numerical_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# 2. Initialize the StandardScaler
scaler = StandardScaler()

# 3. Create copies of your DataFrames to preserve original structures
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 4. Scale ONLY the designated numerical columns
# Fit parameters on the training data slice, then transform both slices
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])


In [45]:
X_train_scaled

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
7832,Europa,False,A,55 Cancri e,-0.267707,False,-0.350087,0.694479,-0.302537,0.300959,0.256747
5842,Europa,False,C,TRAPPIST-1e,0.492810,False,-0.350087,1.252114,2.468142,2.231549,-0.263876
3928,Mars,True,F,55 Cancri e,0.354535,False,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728
4091,Earth,True,G,55 Cancri e,0.561948,False,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728
7679,Earth,True,G,PSO J318.5-22,-0.475120,False,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728
...,...,...,...,...,...,...,...,...,...,...,...
4984,Europa,True,E,55 Cancri e,0.147121,False,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728
6864,Earth,False,F,PSO J318.5-22,-0.475120,False,-0.306405,-0.172758,-0.299193,-0.273653,0.050543
6919,Europa,False,A,TRAPPIST-1e,-0.267707,False,-0.350087,-0.219081,-0.302537,1.357995,3.178542
7137,Europa,False,A,55 Cancri e,0.769362,True,-0.350087,-0.274199,-0.302537,1.071584,0.921373


In [46]:

# OneHotencoding 

# 1. Build the preprocessor without dropping any baseline categories
preprocessor = make_column_transformer(
    (
        
        OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
        make_column_selector(dtype_include=["object", "category"]),
    ),
    (
        # Converts True/False target into 1/0
        FunctionTransformer(
            lambda x: x.astype(int), feature_names_out="one-to-one"
        ),
        ["Transported"],
    ),
    remainder="passthrough",
    verbose_feature_names_out=False,
)

# 2. Transform the data and reconstruct the clean DataFrame
spaceship_clean = pd.DataFrame(
    preprocessor.fit_transform(spaceship_clean),
    columns=preprocessor.get_feature_names_out(),
)



In [49]:
spaceship_clean

,Transported,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,CryoSleep_True,Cabin_A,Cabin_B,Cabin_C,Cabin_D,...,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,24.0,109.0,9.0,25.0,549.0,44.0
2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,58.0,43.0,3576.0,0.0,6715.0,49.0
3,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,33.0,0.0,1283.0,371.0,3329.0,193.0
4,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,16.0,303.0,70.0,151.0,565.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6601,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,41.0,0.0,6819.0,0.0,1643.0,74.0
6602,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,18.0,0.0,0.0,0.0,0.0,0.0
6603,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,26.0,0.0,0.0,1872.0,1.0,0.0
6604,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,32.0,0.0,1049.0,0.0,353.0,3235.0


### Concatenating Scaled training feature and target 

In [53]:
# Combine scaled training features and target labels into a single DataFrame
train_all_in_one = pd.concat([X_train_scaled, pd.Series(y_train, name='Transported', index=X_train_scaled.index)], axis=1)

# Display the first few rows of your full processed dataset
train_all_in_one.head()


,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,CryoSleep_True,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,...,Destination_TRAPPIST-1e,VIP_False,VIP_True,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
5936,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-0.267707,-0.350087,0.694479,-0.302537,0.300959,0.256747,0.0
4416,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.492810,-0.350087,1.252114,2.468142,2.231549,-0.263876,0.0
2948,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.354535,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,1.0
3085,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.561948,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,1.0
5819,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-0.475120,-0.350087,-0.286513,-0.302537,-0.273653,-0.264728,1.0


**Perform Train Test Split**

In [85]:
#from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split

# Perform the train-test split on the single concatenated DataFrame (80% train, 20% test)
df_train_split, df_test_split = train_test_split(train_all_in_one, test_size=0.20, random_state=42)

# Verify the final shapes of your split DataFrames
print(f"Training Data Split Shape: {df_train_split.shape}")
print(f"Testing Data Split Shape:  {df_test_split.shape}")


Training Data Split Shape: (4227, 25)
Testing Data Split Shape:  (1057, 25)


### KNN Model :

In [90]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Initialize the KNN classifier (using standard baseline of 5 neighbors)
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model on your training sets
knn_model.fit(X_train_final, y_train_final)

# Evaluate its accuracy on the testing sets
print(f" KNN Accuracy: {accuracy_score(y_test_final, knn_model.predict(X_test_final))}")


 KNN Accuracy: 0.7767265846736046


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [87]:
#your code here


# Baseline Estimator
base_knn = KNeighborsClassifier(n_neighbors=5)

# 1. BAGGING (With Replacement)
bagging = BaggingClassifier(estimator=base_knn, n_estimators=50, max_samples=0.8, bootstrap=True, random_state=42, n_jobs=-1)
bagging.fit(X_train_final, y_train_final)

# 2. PASTING (Without Replacement)
pasting = BaggingClassifier(estimator=base_knn, n_estimators=50, max_samples=0.8, bootstrap=False, random_state=42, n_jobs=-1)
pasting.fit(X_train_final, y_train_final)

# Evaluation
print(f"Bagging Accuracy: {accuracy_score(y_test_final, bagging.predict(X_test_final))}")
print(f"Pasting  Accuracy: {accuracy_score(y_test_final, pasting.predict(X_test_final))}")


Bagging Accuracy: 0.7691579943235572
Pasting  Accuracy: 0.7663197729422895


- Random Forests

In [77]:
#your code here

# Initialize and train the Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_final, y_train_final)

# Evaluate performance
print(f"🌲 Random Forest Accuracy: {accuracy_score(y_test_final, rf_model.predict(X_test_final))}")


🌲 Random Forest Accuracy: 0.7483443708609272


- Gradient Boosting

In [79]:
#your code here


# Initialize and train Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_final, y_train_final)

# Evaluate performance
print(f" Gradient Boosting Accuracy: {accuracy_score(y_test_final, gb_model.predict(X_test_final))}")


🔥 Gradient Boosting Accuracy: 0.7899716177861873


- Adaptive Boosting

In [83]:
#your code here


# Initialize and train Adaptive Boosting (AdaBoost)
ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada_model.fit(X_train_final, y_train_final)

# Evaluate performance
print(f"AdaBoost Accuracy: {accuracy_score(y_test_final, ada_model.predict(X_test_final))}")


AdaBoost Accuracy: 0.7483443708609272


Which model is the best and why?

In [84]:
#comment here
# Create a dictionary of all the models you trained
models = {
    "K-Nearest Neighbors (KNN)": knn_model,
    "Bagging + KNN": bagging,
    "Pasting + KNN": pasting,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "AdaBoost": ada_model
}

# Print the performance leaderboard
print("🏆 MODEL ACCURACY COMPARISON LEADERBOARD 🏆")
print("-" * 45)
for name, model in models.items():
    acc = accuracy_score(y_test_final, model.predict(X_test_final))
    print(f"| {name:<26} | Accuracy: {acc:.2%} |")
print("-" * 45)


🏆 MODEL ACCURACY COMPARISON LEADERBOARD 🏆
---------------------------------------------
| K-Nearest Neighbors (KNN)  | Accuracy: 77.67% |
| Bagging + KNN              | Accuracy: 76.92% |
| Pasting + KNN              | Accuracy: 76.63% |
| Random Forest              | Accuracy: 74.83% |
| Gradient Boosting          | Accuracy: 79.00% |
| AdaBoost                   | Accuracy: 74.83% |
---------------------------------------------
